In [1]:
import tensorflow as tf
import tf2onnx
import os
VERSION = "V4.5_"

In [ ]:
import tensorflow as tf
import tf2onnx
import numpy as np

# 1. 重新构建并注入权重
new_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(60, 258)), 
    tf.keras.layers.LSTM(64, return_sequences=True, activation='tanh'),
    tf.keras.layers.LSTM(128, return_sequences=True, activation='tanh'),
    tf.keras.layers.LSTM(64, return_sequences=False, activation='tanh'),
    tf.keras.layers.Dense(64, activation='tanh'),
    tf.keras.layers.Dense(32, activation='tanh'),
    tf.keras.layers.Dense(4, activation='softmax')
])

# 刚才注入成功的操作
with np.load('v45_weights_raw.npz') as data:
    loaded_weights = [data[f'arr_{i}'] for i in range(len(data.files))]
new_model.set_weights(loaded_weights)
print("✅ 权重已加载")

# --- 核心转换逻辑开始 ---

# 2. 将 Keras 模型包装成一个 tf.function
@tf.function
def model_func(input_tensor):
    return new_model(input_tensor)

# 3. 创建输入签名
input_signature = [tf.TensorSpec([None, 60, 258], tf.float32, name='input')]

# 4. 使用 from_function 替代 from_keras (避开了 output_names 报错)
onnx_model, _ = tf2onnx.convert.from_function(
    model_func, 
    input_signature=input_signature, 
    opset=13
)

# 5. 保存
onnx_filename = "student_behavior_v4.5_final.onnx"
with open(onnx_filename, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"🚀 转换圆满完成！文件已保存为: {onnx_filename}")

✅ 权重已加载


rewriter <function rewrite_constant_fold at 0x000002C52E426290>: exception `np.cast` was removed in the NumPy 2.0 release. Use `np.asarray(arr, dtype=dtype)` instead.
rewriter <function rewrite_constant_fold at 0x000002C52E426290>: exception `np.cast` was removed in the NumPy 2.0 release. Use `np.asarray(arr, dtype=dtype)` instead.
rewriter <function rewrite_constant_fold at 0x000002C52E426290>: exception `np.cast` was removed in the NumPy 2.0 release. Use `np.asarray(arr, dtype=dtype)` instead.
rewriter <function rewrite_constant_fold at 0x000002C52E426290>: exception `np.cast` was removed in the NumPy 2.0 release. Use `np.asarray(arr, dtype=dtype)` instead.
rewriter <function rewrite_constant_fold at 0x000002C52E426290>: exception `np.cast` was removed in the NumPy 2.0 release. Use `np.asarray(arr, dtype=dtype)` instead.
rewriter <function rewrite_constant_fold at 0x000002C52E426290>: exception `np.cast` was removed in the NumPy 2.0 release. Use `np.asarray(arr, dtype=dtype)` instead

🚀 转换圆满完成！文件已保存为: student_behavior_v4.5_final.onnx
